# Summary

### Data Integrity
- Verified no duplicate listing IDs exist

### Missing Values
- Replaced missing values in `host_is_superhost`, `host_response_time`, and `license` with `Unknown`
- Numerical columns retain null values to avoid skewing distributions in visualizations

### Column Renaming
- `price` → `price_usd` (clarifies currency)
- `neighbourhood_cleansed` → `neighborhood`
- `neighbourhood_group_cleansed` → `neighborhood_group`

### Value Recoding
- `host_is_superhost`, `instant_bookable`: `t`/`f` → `Yes`/`No`
- `host_response_time`: values capitalized for consistent formatting
- `license`: consolidated into `Licensed`, `Exempt`, `Unknown`
- `property_type` → `property_type_detail` (original values preserved)
  - New `property_type` column created, consolidated into 6 categories: `House`, `Apartment`, `Hotel room`, `Private room`, `Shared room`, `Unique`
  - `room_type` dropped due to redundancy with new `property_type`

### Added Columns
- `state` and `country` added for Tableau geocoding compatibility

# Initial Inspection

In [23]:
import pandas as pd

# Ensure all columns are displayed
pd.set_option('display.max_columns', None)

# Load DataFrame
airbnb = pd.read_csv('listings.csv', low_memory=False)

# View dataframe shape and datatypes for each column
airbnb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45533 entries, 0 to 45532
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            45533 non-null  int64  
 1   name                          45532 non-null  object 
 2   host_id                       45533 non-null  int64  
 3   host_name                     45531 non-null  object 
 4   host_since                    45531 non-null  object 
 5   host_response_time            35445 non-null  object 
 6   host_response_rate            35445 non-null  float64
 7   host_is_superhost             44281 non-null  object 
 8   neighbourhood_cleansed        45533 non-null  object 
 9   neighbourhood_group_cleansed  45533 non-null  object 
 10  latitude                      45533 non-null  float64
 11  longitude                     45533 non-null  float64
 12  property_type                 45533 non-null  object 
 13  r

In [24]:
# Preview dataframe
airbnb.head(3)

,id,name,host_id,host_name,host_since,host_response_time,host_response_rate,host_is_superhost,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,availability_365,number_of_reviews,review_scores_rating,license,instant_bookable
0,670339032744709144,Westwood lovely three bedrooms three bathrooms,4780152,Moon,20/01/13,within a few hours,0.96,f,West Los Angeles,City of Los Angeles,34.049660,-118.435550,Entire condo,Entire home/apt,6,3.0,3.0,3.0,399.0,30,365,0,NaN,NaN,f
1,37014494,Spanish style lower duplex near Beverly Hills,278288178,Ida,22/07/19,NaN,NaN,f,Beverlywood,City of Los Angeles,34.048410,-118.387510,Entire rental unit,Entire home/apt,2,NaN,2.0,NaN,NaN,30,0,0,NaN,NaN,f
2,1024835174766068422,Charming Beverly Hills Home,513813179,Tiana,08/05/23,within a day,0.60,f,Beverly Hills,Other Cities,34.070583,-118.390742,Entire home,Entire home/apt,6,3.0,3.0,3.0,434.0,30,267,0,NaN,NaN,f


## Checking for Duplicates

In [25]:
# Check for duplicate IDs
print(f"Duplicate IDs: {airbnb.duplicated('id').sum()}")

Duplicate IDs: 0


## Checking for Missing Values

In [26]:
# Show number of missing values
airbnb.isna().sum()[airbnb.isna().sum() > 0]

name                        1
host_name                   2
host_since                  2
host_response_time      10088
host_response_rate      10088
host_is_superhost        1252
bathrooms                8239
bedrooms                 3039
beds                     8334
price                    8237
review_scores_rating    12146
license                 32730
dtype: int64

In [27]:
# Drop rows with 1-2 missing values
airbnb = airbnb.dropna(subset=['name', 'host_name', 'host_since']).reset_index(drop=True)

# Fill missing values in categorical columns
cols_unknown = ['host_is_superhost', 'host_response_time']
airbnb[cols_unknown] = airbnb[cols_unknown].fillna("Unknown")

# Recategorize 'license' column for clarity
def license_type(val):

    val_lower = str(val).lower()
    
    if pd.isna(val):
        return "Unknown"
    elif "exempt" in val_lower:
        return "Exempt"
    else:
        return "Licensed"

airbnb['license'] = airbnb['license'].apply(license_type)

# Confirm missing values in categorical columns are filled
print(f"Missing values remaining: {airbnb[['host_is_superhost', 'host_response_time', 'license']].isna().sum().sum()}")

Missing values remaining: 0


## Additional Formatting

In [28]:
# Recategorize 'property_type'
def cat_property(val):
    if pd.isna(val):
        return
        
    val_lower = val.lower()
    
    if 'private room' in val_lower:
        return "Private room"
    elif any(x in val_lower for x in ['shared', 'hostel']):
        return "Shared room"
    elif any(x in val_lower for x in ['hotel', 'breakfast']):
        return "Hotel room"
    elif any(x in val_lower for x in ['apartment', 'condo', 'loft', 'serviced', 'suite', 'unit']):
        return "Apartment"
    elif any(x in val_lower for x in ['cycladic', 'earthen', 'farm', 'tiny', 'tree']):
        return "Unique"
    elif any(x in val_lower for x in ['bungalow', 'cabin', 'casa', 'cottage', 'home', 'house', 'place', 'villa']):
        return "House"
    else:
        return "Unique"

airbnb = airbnb.rename(columns={'property_type': 'property_type_detail'})
airbnb = airbnb.drop(columns='room_type')

col_index = airbnb.columns.tolist().index('property_type_detail')
airbnb.insert(loc=col_index, column='property_type', value=airbnb['property_type_detail'].apply(cat_property))


In [29]:
# Rename values for clarity
tf_rename = ['host_is_superhost', 'instant_bookable']
airbnb[tf_rename] = airbnb[tf_rename].replace({"t":"Yes", "f":"No"})

airbnb['price'] = airbnb['price'].round(2)
airbnb = airbnb.rename(columns={'price': 'price_usd', 'neighbourhood_cleansed': 'neighborhood', 'neighbourhood_group_cleansed': 'neighborhood_group'})

# Convert bedrooms and beds to dtype int for clarity
cols_int = ['bedrooms', 'beds']
airbnb[cols_int] = airbnb[cols_int].astype('Int64')

# Convert dates to dtype datetime for streamlined analysis
def parse_date(val): 
    try:
        # Covert normally
        return pd.to_datetime(val, dayfirst=True)
    except:
        # Convert any Excel serial numbers found
        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(float(val)))

airbnb['host_since'] = airbnb['host_since'].apply(parse_date)

In [30]:
# Add state and country values for Tableau geocoding
airbnb.columns.tolist()
airbnb.insert(loc=10, column='state', value='CA')
airbnb.insert(loc=11, column='country', value='US')

In [31]:
# Fix value formats for consistent capitalization
airbnb['host_response_time'] = airbnb['host_response_time'].str.capitalize()

### Adding Median Income Column

In [32]:
# Map for zipcodes with income
income_map = {
    'Acton': ['93510', 90500],
    'Adams-Normandie': ['90007', 95500],
    'Agoura Hills': ['91301', 169600],
    'Agua Dulce': ['91390', 93000],
    'Alhambra': ['91801', 92000],
    'Alondra Park': ['90260', 94000],
    'Altadena': ['91001', 92000],
    'Angeles Crest': ['91208', 170400],
    'Arcadia': ['91006', 91500],
    'Arleta': ['91331', 91000],
    'Arlington Heights': ['90019', 96500],
    'Artesia': ['90701', 91500],
    'Athens': ['90044', 54400],
    'Atwater Village': ['90039', 95500],
    'Avalon': ['90704', 91000],
    'Avocado Heights': ['91746', 95500],
    'Azusa': ['91702', 90500],
    'Baldwin Hills/Crenshaw': ['90008', 177600],
    'Baldwin Park': ['91706', 94000],
    'Bel-Air': ['90077', 165600],
    'Bell': ['90201', 90000],
    'Bell Gardens': ['90201', 94000],
    'Bellflower': ['90706', 93000],
    'Beverly Crest': ['90210', 170400],
    'Beverly Grove': ['90048', 94500],
    'Beverly Hills': ['90210', 132977],
    'Beverlywood': ['90035', 93500],
    'Boyle Heights': ['90033', 57200],
    'Brentwood': ['90049', 167200],
    'Broadway-Manchester': ['90003', 97500],
    'Burbank': ['91502', 97082],
    'Calabasas': ['91302', 167200],
    'Canoga Park': ['91303', 93500],
    'Carson': ['90745', 91000],
    'Carthay': ['90035', 91500],
    'Castaic': ['91384', 91500],
    'Castaic Canyons': ['91384', 95500],
    'Central-Alameda': ['90011', 95500],
    'Century City': ['90067', 94000],
    'Cerritos': ['90703', 92000],
    'Charter Oak': ['91724', 93500],
    'Chatsworth': ['91311', 93000],
    'Chatsworth Reservoir': ['91311', 98000],
    'Chesterfield Square': ['90062', 97500],
    'Cheviot Hills': ['90064', 170400],
    'Chinatown': ['90012', 92500],
    'Citrus': ['91722', 91000],
    'Claremont': ['91711', 92500],
    'Commerce': ['90040', 92000],
    'Compton': ['90220', 78465],
    'Covina': ['91722', 91000],
    'Cudahy': ['90201', 91000],
    'Culver City': ['90230', 93500],
    'Cypress Park': ['90065', 94000],
    'Del Aire': ['90250', 92000],
    'Del Rey': ['90066', 91500],
    'Desert View Highlands': ['93551', 98500],
    'Diamond Bar': ['91765', 93500],
    'Downey': ['90240', 91000],
    'Downtown': ['90012', 92000],
    'Duarte': ['91010', 91000],
    'Eagle Rock': ['90041', 93000],
    'East Compton': ['90221', 56800],
    'East Hollywood': ['90029', 95000],
    'East Los Angeles': ['90022', 96000],
    'East Pasadena': ['91107', 94500],
    'East San Gabriel': ['91775', 96000],
    'East Whittier': ['90602', 94500],
    'Echo Park': ['90026', 92500],
    'El Monte': ['91731', 92000],
    'El Segundo': ['90245', 93000],
    'El Sereno': ['90032', 92500],
    'Elizabeth Lake': ['93532', 95000],
    'Elysian Park': ['90012', 94000],
    'Elysian Valley': ['90039', 95000],
    'Encino': ['91316', 164800],
    'Exposition Park': ['90007', 95500],
    'Fairfax': ['90036', 91500],
    'Florence': ['90001', 55200],
    'Florence-Firestone': ['90001', 59200],
    'Gardena': ['90247', 91500],
    'Glassell Park': ['90065', 94500],
    'Glendale': ['91201', 92000],
    'Glendora': ['91740', 92000],
    'Gramercy Park': ['90047', 94500],
    'Granada Hills': ['91344', 170400],
    'Green Meadows': ['90003', 94500],
    'Green Valley': ['91390', 94000],
    'Griffith Park': ['90027', 94500],
    'Hacienda Heights': ['91745', 96000],
    'Hancock Park': ['90004', 94000],
    'Harbor City': ['90710', 93500],
    'Harbor Gateway': ['90248', 95000],
    'Harvard Heights': ['90006', 95500],
    'Harvard Park': ['90047', 94000],
    'Hasley Canyon': ['91384', 94500],
    'Hawaiian Gardens': ['90716', 96000],
    'Hawthorne': ['90250', 92500],
    'Hermosa Beach': ['90254', 170400],
    'Hidden Hills': ['91302', 169600],
    'Highland Park': ['90042', 94500],
    'Historic South-Central': ['90011', 60800],
    'Hollywood': ['90028', 92500],
    'Hollywood Hills': ['90068', 172000],
    'Hollywood Hills West': ['90069', 176000],
    'Huntington Park': ['90255', 58000],
    'Hyde Park': ['90043', 92500],
    'Industry': ['91746', 92000],
    'Inglewood': ['90301', 92500],
    'Irwindale': ['91706', 92500],
    'Jefferson Park': ['90018', 95000],
    'Koreatown': ['90005', 92500],
    'La Canada Flintridge': ['91011', 176000],
    'La Crescenta-Montrose': ['91214', 98500],
    'La Habra Heights': ['90631', 96000],
    'La Mirada': ['90638', 92500],
    'La Puente': ['91744', 92500],
    'La Verne': ['91750', 92000],
    'Ladera Heights': ['90056', 95000],
    'Lake Balboa': ['91406', 93500],
    'Lake Hughes': ['93532', 93500],
    'Lake Los Angeles': ['93535', 96000],
    'Lake View Terrace': ['91342', 96500],
    'Lakewood': ['90712', 92000],
    'Lancaster': ['93534', 92500],
    'Larchmont': ['90004', 92500],
    'Lawndale': ['90260', 92000],
    'Leimert Park': ['90008', 94000],
    'Lennox': ['90304', 91000],
    'Leona Valley': ['93551', 94000],
    'Lincoln Heights': ['90031', 95500],
    'Lomita': ['90717', 91000],
    'Long Beach': ['90802', 75000],
    'Lopez/Kagel Canyons': ['91342', 97500],
    'Los Feliz': ['90027', 92500],
    'Lynwood': ['90262', 54800],
    'Malibu': ['90265', 164800],
    'Manchester Square': ['90047', 96500],
    'Manhattan Beach': ['90266', 172000],
    'Mar Vista': ['90066', 92500],
    'Marina del Rey': ['90292', 95000],
    'Mayflower Village': ['91706', 96500],
    'Maywood': ['90270', 54800],
    'Mid-City': ['90019', 92000],
    'Mid-Wilshire': ['90036', 94000],
    'Mission Hills': ['91345', 170400],
    'Monrovia': ['91016', 92000],
    'Montebello': ['90640', 93000],
    'Montecito Heights': ['90031', 96500],
    'Monterey Park': ['91754', 94500],
    'Mount Washington': ['90065', 96000],
    'North El Monte': ['91732', 95000],
    'North Hills': ['91343', 168800],
    'North Hollywood': ['91601', 95500],
    'North Whittier': ['90601', 95000],
    'Northeast Antelope Valley': ['93535', 100500],
    'Northridge': ['91324', 93000],
    'Northwest Antelope Valley': ['93536', 100500],
    'Northwest Palmdale': ['93551', 97000],
    'Norwalk': ['90650', 91500],
    'Pacific Palisades': ['90272', 173600],
    'Pacoima': ['91331', 54800],
    'Palmdale': ['93550', 92000],
    'Palms': ['90034', 90500],
    'Palos Verdes Estates': ['90274', 176000],
    'Panorama City': ['91402', 94500],
    'Paramount': ['90723', 92500],
    'Pasadena': ['91101', 92000],
    'Pico Rivera': ['90660', 93500],
    'Pico-Robertson': ['90035', 95000],
    'Pico-Union': ['90006', 93000],
    'Playa Vista': ['90094', 93500],
    'Playa del Rey': ['90293', 94500],
    'Pomona': ['91766', 91000],
    'Porter Ranch': ['91326', 94000],
    'Quartz Hill': ['93536', 93500],
    'Rancho Dominguez': ['90220', 96000],
    'Rancho Palos Verdes': ['90275', 175200],
    'Rancho Park': ['90064', 93500],
    'Redondo Beach': ['90277', 170400],
    'Reseda': ['91335', 91000],
    'Ridge Route': ['91384', 93500],
    'Rolling Hills': ['90274', 170400],
    'Rolling Hills Estates': ['90274', 176800],
    'Rosemead': ['91770', 92000],
    'Rowland Heights': ['91748', 95500],
    'San Dimas': ['91773', 92500],
    'San Fernando': ['91340', 94000],
    'San Gabriel': ['91776', 93500],
    'San Marino': ['91108', 168000],
    'San Pasqual': ['91107', 93500],
    'San Pedro': ['90731', 92500],
    'Santa Clarita': ['91350', 94500],
    'Santa Fe Springs': ['90670', 96000],
    'Santa Monica': ['90401', 110000],
    'Sawtelle': ['90025', 92000],
    'Sepulveda Basin': ['91406', 95500],
    'Shadow Hills': ['91040', 169600],
    'Sherman Oaks': ['91403', 169600],
    'Sierra Madre': ['91024', 94000],
    'Signal Hill': ['90755', 93500],
    'Silver Lake': ['90026', 93500],
    'South Diamond Bar': ['91765', 96500],
    'South El Monte': ['91733', 95000],
    'South Gate': ['90280', 93000],
    'South Park': ['90011', 93000],
    'South Pasadena': ['91030', 95000],
    'South San Gabriel': ['91770', 96500],
    'South San Jose Hills': ['91744', 176000],
    'South Whittier': ['90605', 95000],
    'Southeast Antelope Valley': ['93552', 100500],
    'Stevenson Ranch': ['91381', 95500],
    'Studio City': ['91604', 93500],
    'Sun Valley': ['91352', 93000],
    'Sun Village': ['93543', 93500],
    'Sunland': ['91040', 91500],
    'Sylmar': ['91342', 91000],
    'Tarzana': ['91356', 91500],
    'Temple City': ['91780', 93500],
    'Toluca Lake': ['91602', 93500],
    'Topanga': ['90290', 91500],
    'Torrance': ['90501', 92000],
    'Tujunga': ['91042', 91500],
    'Tujunga Canyons': ['91042', 95500],
    'Unincorporated Catalina Island': ['90704', 103000],
    'Unincorporated Santa Monica Mountains': ['90290', 106500],
    'Unincorporated Santa Susana Mountains': ['91311', 106500],
    'Universal City': ['91608', 95000],
    'University Park': ['90007', 95500],
    'Val Verde': ['91384', 92500],
    'Valinda': ['91744', 91500],
    'Valley Glen': ['91401', 93500],
    'Valley Village': ['91607', 95000],
    'Van Nuys': ['91401', 92000],
    'Venice': ['90291', 91000],
    'Vermont Knolls': ['90044', 57600],
    'Vermont Square': ['90037', 57600],
    'Vermont Vista': ['90044', 57200],
    'Vermont-Slauson': ['90044', 58000],
    'Vernon': ['90058', 54400],
    'Veterans Administration': ['90073', 99500],
    'View Park-Windsor Hills': ['90043', 178400],
    'Vincent': ['91722', 91500],
    'Walnut': ['91789', 91000],
    'Walnut Park': ['90255', 93500],
    'Watts': ['90002', 54000],
    'West Adams': ['90016', 93000],
    'West Carson': ['90745', 93500],
    'West Compton': ['90220', 56800],
    'West Covina': ['91790', 93500],
    'West Hills': ['91307', 168000],
    'West Hollywood': ['90069', 95000],
    'West Los Angeles': ['90025', 96000],
    'West Puente Valley': ['91744', 97000],
    'West Whittier-Los Nietos': ['90606', 100000],
    'Westchester': ['90045', 93500],
    'Westlake': ['90057', 92000],
    'Westlake Village': ['91361', 96000],
    'Westmont': ['90044', 92000],
    'Westwood': ['90024', 166400],
    'Whittier': ['90601', 92000],
    'Willowbrook': ['90222', 93500],
    'Wilmington': ['90744', 93000],
    'Windsor Square': ['90004', 95000],
    'Winnetka': ['91306', 92000],
    'Woodland Hills': ['91364', 171200]
}

# Add column for neighborhood zipcodes
airbnb['neighborhood_zipcode'] = airbnb['neighborhood'].map(lambda x: income_map.get(x)[0])

# Add column for neighborhood median incomes
airbnb['neighborhood_median_income'] = airbnb['neighborhood'].map(lambda x: income_map.get(x)[1])

## Final Preview

In [33]:
# Updated preview
airbnb.head(3)

,id,name,host_id,host_name,host_since,host_response_time,host_response_rate,host_is_superhost,neighborhood,neighborhood_group,state,country,latitude,longitude,property_type,property_type_detail,accommodates,bathrooms,bedrooms,beds,price_usd,minimum_nights,availability_365,number_of_reviews,review_scores_rating,license,instant_bookable,neighborhood_zipcode,neighborhood_median_income
0,670339032744709144,Westwood lovely three bedrooms three bathrooms,4780152,Moon,2013-01-20,Within a few hours,0.96,No,West Los Angeles,City of Los Angeles,CA,US,34.049660,-118.435550,Apartment,Entire condo,6,3.0,3,3,399.0,30,365,0,NaN,Unknown,No,90025,96000
1,37014494,Spanish style lower duplex near Beverly Hills,278288178,Ida,2019-07-22,Unknown,NaN,No,Beverlywood,City of Los Angeles,CA,US,34.048410,-118.387510,Apartment,Entire rental unit,2,NaN,2,<NA>,NaN,30,0,0,NaN,Unknown,No,90035,93500
2,1024835174766068422,Charming Beverly Hills Home,513813179,Tiana,2023-05-08,Within a day,0.60,No,Beverly Hills,Other Cities,CA,US,34.070583,-118.390742,House,Entire home,6,3.0,3,3,434.0,30,267,0,NaN,Unknown,No,90210,132977


In [34]:
# airbnb.to_csv('airbnb_data_cleaned_v2_with_income.csv', index=False)